### 멀티모달 모델
기존의 LLM이 텍스트라는 단일 모드만 처리했다면, 멀티모달 모델은 여러 가지 형태의 정보를 함께 처리할 수 있다.

1. CLIP (Contrastive Language-Image Pre-training)
- OpenAI에서 발표한 모델로, 텍스트와 이미지를 하나의 공간에서 연결하는 혁신적인 방법론이다.
- 기존 AI가 이미지를 "강아지", "고양이"라는 정해진 클래스로만 배웠다면, CLIP은 자연어 설명을 통해 이미지의 맥락을 배운다.

2. 공동 임베딩 공간(Shared Embedding Space)
- 멀티모달 모델은 텍스트와 이미지를 각각의 특징 벡터로 변환한 뒤, 이를 하나의 좌표 평면에 배치한다.
- 텍스트 인코더: "해변을 달리는 골든 리트리버"라는 문장을 벡터로 변환.
- 이미지 인코더: 실제 강아지 사진을 벡터로 변환.
- 대조 학습(Contrastive Learning): 관련 있는 텍스트와 이미지 벡터는 가까워지도록 유도하고, 관련 없는 데이터끼리는 멀어지도록 학습한다.

`관련 문서: https://contents.premium.naver.com/byline/commercebn/contents/260421165257027nn`

3. 벡터 유사도 (Vector Similarity)
- AI는 두 데이터가 얼마나 비슷한지 수학적으로 계산하며, 주로 주로 코사인 유사도(Cosine Similarity)를 사용한다.
- 유사도 1에 가까움: 이 텍스트는 이미지의 정확한 설명이다.
- 유사도 0에 가까움: 이 텍스트와 이미지는 아무 상관이 없다.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
def stream_response(stream):
    for chunk in stream:
        print(chunk.content, end="", flush=True)

In [ ]:
import os
import base64
from langchain_core.messages import HumanMessage

class SimpleMultiModal:
    def __init__(self, llm):
        self.llm = llm

    def stream(self, input_source, prompt="이 이미지를 설명해줘"):
        # 로컬 파일인지 확인
        if os.path.exists(input_source):
            with open(input_source, 'rb') as f:
                data = base64.b64encode(f.read()).decode("utf-8")
                image_url = f'data:image/jpeg;base64,{data}'
        else:
            image_url = input_source

        message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}}
        ])

        return self.llm.stream([message])        

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5.4-nano"
)

smm = SimpleMultiModal(llm)
stream = smm.stream('./images/20251011_152924911_iOS.png', prompt="이 사람이 누구인거같아?")
stream_response(stream)